# Timeseries classification with a Transformer model

**Author:** [Theodoros Ntakouris](https://github.com/ntakouris)<br>
**Anotated by (for CMPE 401):** [Musab Hassan](https://github.com/Musab-Hassan)<br>
**Date created:** 2021/06/25<br>
**Last modified:** 2026/03/12<br>
**Description:** This notebook demonstrates how to do timeseries classification using a Transformer model.

## Introduction

This is the Transformer architecture from
[Attention Is All You Need](https://arxiv.org/abs/1706.03762),
applied to timeseries instead of natural language.

This example requires TensorFlow 2.4 or higher.

## Load the dataset

We are going to use the same dataset and preprocessing as the
[TimeSeries Classification from Scratch](https://keras.io/examples/timeseries/timeseries_classification_from_scratch)
example.

In [14]:
import numpy as np
import keras
import gc
import pandas as pd
from keras import layers


def readucr(filename):
    data = np.loadtxt(filename, delimiter="\t")
    y = data[:, 0]
    x = data[:, 1:]
    return x, y.astype(int) 


root_url = "https://raw.githubusercontent.com/hfawaz/cd-diagram/master/FordA/"

x_train, y_train = readucr(root_url + "FordA_TRAIN.tsv")
x_test, y_test = readucr(root_url + "FordA_TEST.tsv")

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

n_classes = len(np.unique(y_train))

idx = np.random.permutation(len(x_train))
x_train = x_train[idx]
y_train = y_train[idx]

y_train[y_train == -1] = 0
y_test[y_test == -1] = 0

## Build the model

Our model processes a tensor of shape `(batch size, sequence length, features)`,
where `sequence length` is the number of time steps and `features` is each input
timeseries.

You can replace your classification RNN layers with this one: the
inputs are fully compatible!

We include residual connections, layer normalization, and dropout.
The resulting layer can be stacked multiple times.

The projection layers are implemented through `keras.layers.Conv1D`.

In [15]:
# This implementation applies Layer Normalization before the residual connection
# to improve training stability by producing better-behaved gradients and often
# eliminating the need for learning rate warm-up.


def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

The main part of our model is now complete. We can stack multiple of those
`transformer_encoder` blocks and we can also proceed to add the final
Multi-Layer Perceptron classification head. Apart from a stack of `Dense`
layers, we need to reduce the output tensor of the `TransformerEncoder` part of
our model down to a vector of features for each data point in the current
batch. A common way to achieve this is to use a pooling layer. For
this example, a `GlobalAveragePooling1D` layer is sufficient.

In [16]:
def build_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_transformer_blocks,
    mlp_units,
    dropout=0,
    mlp_dropout=0,
):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.GlobalAveragePooling1D(data_format="channels_last")(x)
    for dim in mlp_units:
        x = layers.Dense(dim, activation="relu")(x)
        x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)

## Train and evaluate

In [17]:
input_shape = x_train.shape[1:]

model = build_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=4,
    mlp_units=[128],
    mlp_dropout=0.4,
    dropout=0.25,
)

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)
model.summary()

callbacks = [keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]

model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=64,
    callbacks=callbacks,
)

model.evaluate(x_test, y_test, verbose=1)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 500, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 500, 1)    │      7,169 │ input_layer[0][0… │
│ (MultiHeadAttentio… │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 500, 1)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 500, 1)    │          2 │ dropout_1[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 4)    │          8 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 500, 4)    │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 500, 1)    │          5 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ conv1d_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 500, 1)    │      7,169 │ add_1[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 500, 1)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ dropout_4[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 500, 4)    │          8 │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 500, 4)    │          0 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 500, 1)    │          5 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 500, 1)    │          2 │ conv1d_3[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 500, 1)    │          0 │ layer_normalizat… │
│                     │                   │            │ add_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 29,258 (114.29 KB)

 Trainable params: 29,258 (114.29 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
 1/45 ━━━━━━━━━━━━━━━━━━━━ 6:16 9s/step - loss: 0.6931 - sparse_categorical_accuracy: 0.6094

2026-03-23 16:48:21.692630: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_7', 136 bytes spill stores, 116 bytes spill loads



45/45 ━━━━━━━━━━━━━━━━━━━━ 15s 156ms/step - loss: 0.6931 - sparse_categorical_accuracy: 0.5163 - val_loss: 0.6932 - val_sparse_categorical_accuracy: 0.4979
Epoch 2/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - loss: 0.6930 - sparse_categorical_accuracy: 0.5163 - val_loss: 0.6932 - val_sparse_categorical_accuracy: 0.4979
Epoch 3/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - loss: 0.6929 - sparse_categorical_accuracy: 0.5163 - val_loss: 0.6932 - val_sparse_categorical_accuracy: 0.4979
Epoch 4/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 111ms/step - loss: 0.6928 - sparse_categorical_accuracy: 0.5163 - val_loss: 0.6933 - val_sparse_categorical_accuracy: 0.4979
Epoch 5/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 109ms/step - loss: 0.6927 - sparse_categorical_accuracy: 0.5163 - val_loss: 0.6933 - val_sparse_categorical_accuracy: 0.4979
Epoch 6/150
45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - loss: 0.6928 - sparse_categorical_accuracy: 0.5163 - val_loss: 0.6934 - val_sparse_categorical_accuracy: 0.4979
Epoch 7/1

[0.6930089592933655, 0.5159090757369995]

## Modifications (Task 2)


In [18]:
# Store baseline results
test_acc_baseline = model.evaluate(x_test, y_test, verbose=0)[1]
experiments_results = [{"name": "Baseline", "num_blocks": 4, "num_heads": 4, "dropout": 0.25, "test_acc": test_acc_baseline, "ff_dim": 4}]
print(f"Baseline test accuracy: {test_acc_baseline:.4f}")

# Free memory
del model
keras.backend.clear_session()
gc.collect()

Baseline test accuracy: 0.5159


0

In [19]:
# Modification 1: Reduce Dropout
model_v1 = build_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=4,
    mlp_units=[128],
    mlp_dropout=0.2,
    dropout=0.1,  # Reduced from 0.25
)

model_v1.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)

model_v1.fit(x_train, y_train, validation_split=0.2, epochs=150, batch_size=64, callbacks=callbacks, verbose=0)
test_acc_v1 = model_v1.evaluate(x_test, y_test, verbose=0)[1]

print(f"Mod 1 (Lower Dropout) test accuracy: {test_acc_v1:.4f}")
experiments_results.append({"name": "Mod 1: Lower Dropout", "num_blocks": 4, "num_heads": 4, "dropout": 0.1, "test_acc": test_acc_v1, "ff_dim": 4})

2026-03-23 16:49:30.785852: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_7', 136 bytes spill stores, 116 bytes spill loads



Mod 1 (Lower Dropout) test accuracy: 0.5159


In [20]:
# Free memory
del model_v1
keras.backend.clear_session()
gc.collect()

0

In [21]:
# Modification 2: Reduce Feed-Forward Size
model_v2 = build_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=2,  # Reduced from 4
    num_transformer_blocks=4,
    mlp_units=[128],
    mlp_dropout=0.4,
    dropout=0.25,
)

model_v2.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)

model_v2.fit(x_train, y_train, validation_split=0.2, epochs=150, batch_size=64, callbacks=callbacks, verbose=0)
test_acc_v2 = model_v2.evaluate(x_test, y_test, verbose=0)[1]

print(f"Mod 2 (Smaller FF Dim) test accuracy: {test_acc_v2:.4f}")
experiments_results.append({"name": "Mod 2: Smaller FF Dim", "ff_dim": 2, "test_acc": test_acc_v2, "num_blocks": 4, "num_heads": 4, "dropout": 0.25})

2026-03-23 16:50:41.170451: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_7', 136 bytes spill stores, 116 bytes spill loads



Mod 2 (Smaller FF Dim) test accuracy: 0.5159


In [22]:
# Free memory
del model_v2
keras.backend.clear_session()
gc.collect()

0

In [24]:
# Modification 3: Take out two transformer blocks
model_v3 = build_model(
    input_shape,
    head_size=256,
    num_heads=4,
    ff_dim=4,
    num_transformer_blocks=2, # Decreased from 4
    mlp_units=[128],
    mlp_dropout=0.4,
    dropout=0.25,
)

model_v3.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)

model_v3.fit(x_train, y_train, validation_split=0.2, epochs=150, batch_size=64, callbacks=callbacks, verbose=0)
test_acc_v3 = model_v3.evaluate(x_test, y_test, verbose=0)[1]

print(f"Mod 3 (Fewer Transformer Blocks) test accuracy: {test_acc_v3:.4f}")
experiments_results.append({"name": "Mod 3: Fewer Transformer Blocks", "num_blocks": 2, "num_heads": 4, "dropout": 0.25, "ff_dim": 4, "test_acc": test_acc_v3})

2026-03-23 16:52:44.598950: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_transpose_fusion_3', 136 bytes spill stores, 116 bytes spill loads



Mod 3 (Fewer Transformer Blocks) test accuracy: 0.5159


In [25]:
# Free memory
del model_v3
keras.backend.clear_session()
gc.collect()

0

In [26]:
# Display results summary
results_df = pd.DataFrame(experiments_results)
print(results_df.to_string(index=False))

                           name  num_blocks  num_heads  dropout  test_acc  ff_dim
                       Baseline           4          4     0.25  0.515909       4
           Mod 1: Lower Dropout           4          4     0.10  0.515909       4
          Mod 2: Smaller FF Dim           4          4     0.25  0.515909       2
Mod 3: Fewer Transformer Blocks           2          4     0.25  0.515909       4
